# Please tell me you know how to find all the CPTAC data

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** specific counts below are still placeholders -- not yet run against the live API. The drs_uri explanation and structure have been reviewed and corrected; counts need verification.

I just found out that the Clinical Proteomic Tumor Analysis Consortium (CPTAC) exists, and that its data got spread out across several different data centers -- not because anyone organized it that way, but because that's just where pieces of it happened to land over the years. Every one of those data centers has its own format, its own identifiers, its own way of describing things. If I had to go find every piece myself and figure out which subject in one repository was the same person as which subject in another, that's not a weekend project. That's a years-long archival nightmare, and I don't have years.

So before I give up on reusing this data entirely, I want to know: can this tool actually do that part for me?

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

In [ ]:
columns(column=['*project*'])

`project_name` looks like the right field. I'm searching for people, not files, so:

In [ ]:
summarize_subjects(match_all='project_name = *cptac*')

That's it. One line. Every subject tagged as part of CPTAC, already reconciled across however many data centers their pieces actually live in, already matched up as the same people, without me ever touching an identifier-matching problem by hand. That multi-year harmonization headache I was dreading is just... done, invisibly, before I even ran this cell.

I'll want the files eventually too, so let's see what exists:

In [ ]:
summarize_files(match_all='project_name = *cptac*')

That's a lot of files, so let me narrow to something specific -- I'm interested in kidney and bladder patients today. I still want everything to be CPTAC, *and* I want it to touch one of two anatomic sites, so I need an "all of this" condition and an "any of these" condition at the same time:

In [ ]:
summarize_files(
    match_all=['project_name = *cptac*'],
    match_any=['anatomic_site = *kidney*', 'anatomic_site = *bladder*']
)

Since I'll actually need to retrieve these files eventually, I want the `drs_uri` for each one:

In [ ]:
get_file_data(
    match_all=['project_name = *cptac*'],
    match_any=['anatomic_site = *kidney*', 'anatomic_site = *bladder*']
)

A `drs_uri` points directly at the cloud bucket where a file actually lives -- not at some portal or interface I'd have to visit at GDC, PDC, or wherever the file originally landed. I can hand these URIs straight to a cloud workspace like Terra or the Cancer Genomics Cloud, and the workspace resolves them on its own. There's no detour through any individual data center's own system. The only real constraint is access: if a file is controlled-access and I don't have the right permissions, having the URI doesn't get me the file -- but for anything open-access, this is the entire retrieval step.

One thing worth knowing before I start slicing this by data center: the per-source numbers won't add up to my total the way I'd expect. If I check how many of my CPTAC subjects come from just one source --

In [ ]:
summarize_subjects(match_all='project_name = *cptac*', data_source='GDC')

-- and compare that to my total from earlier, the difference isn't just "the rest came from elsewhere." A lot of these subjects have data at *more than one* data center, so they're counted once in my combined total but would also show up if I checked any other single source they happen to live in too. That's not a bug or a double-count on CDA's part -- it's just what cross-source reassembly actually looks like once you're reconnecting a project that was scattered to begin with.